# EDA: Healthcare Admissions Dataset

This is the analysis that informed the normalized schema in `app/database/models.py` -- see `docs/DATABASE_SCHEMA.md` for the write-up. Run from the repo root with the `app` package importable.

In [ ]:
import pandas as pd

df = pd.read_csv('../data/raw/healthcare_dataset.csv')
print(df.shape)
df.head()

## Cardinality check -- which columns are dimension-table candidates?

In [ ]:
for col in ['Name', 'Doctor', 'Hospital', 'Insurance Provider', 'Medical Condition', 'Admission Type', 'Test Results']:
    print(f"{col:22s} nunique={df[col].nunique():>7} / {len(df)} rows")

`Doctor` and `Hospital` are nearly as high-cardinality as the row count itself -- normalizing them still helps (indexed FK vs. VARCHAR scan, no repeated string storage) but they are not meaningfully many-to-few like `Insurance Provider` (5 distinct values) or `Medical Condition` (6 distinct values). `Name` is high-cardinality with likely collisions across unrelated admissions -- see `docs/DATABASE_SCHEMA.md` for why this rules out a `patients` dimension table keyed by name.

In [ ]:
print('Duplicate name collisions (same name appears on >1 admission):')
print((df['Name'].value_counts() > 1).sum(), 'names repeat')
print(df.isna().sum())
print(df.duplicated().sum(), 'exact duplicate rows')

## Enum-like columns and their exact values (used to ground the NL2SQL prompt)

In [ ]:
for col in ['Medical Condition', 'Admission Type', 'Test Results', 'Gender', 'Blood Type']:
    print(col, ':', sorted(df[col].unique().tolist()))